# CodeForge ASM Autoresearch

Trains Ministral-8B to generate correct NASM x86-64 assembly via GRPO.

**Setup:** Runtime > Change runtime type > T4 (free) or A100 (Pro)  
**Secrets needed:** `CLAUDE_CODE_OAUTH_TOKEN`, `HF_TOKEN` (for Ministral-8B)

## 1. Auth

In [ ]:
import os
from google.colab import userdata

os.environ["CLAUDE_CODE_OAUTH_TOKEN"] = userdata.get("CLAUDE_CODE_OAUTH_TOKEN")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ.pop("ANTHROPIC_API_KEY", None)

print("CLAUDE token:", os.environ["CLAUDE_CODE_OAUTH_TOKEN"][:12] + "...")
print("HF token:", os.environ["HF_TOKEN"][:12] + "...")

## 2. Instalar Claude Code CLI

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs -q
!npm install -g @anthropic-ai/claude-code --silent
!claude --version

## 3. Upload do projeto

Compacte `codeforge-asm/` em zip e selecione quando aparecer o seletor.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
!unzip -q "{zip_name}" -d /content/
!ls /content/codeforge-asm/

## 4. Instalar dependencias

In [ ]:
import os
%cd /content/codeforge-asm

# System deps (nasm + ld)
!apt-get install -y nasm binutils -q
!nasm --version && ld --version | head -1

# Python deps
!pip install -q -r requirements.txt

print("Done!")

## 5. Preparar workspace

In [ ]:
%cd /content/codeforge-asm
!git config user.email "autoresearch@colab.local"
!git config user.name "AutoResearch Agent"
!python prepare.py

## 6. Teste rapido (dry_run)

Verifica o pipeline sem gastar GPU.

In [ ]:
%cd /content/codeforge-asm
# Roda 1 iteracao com dry_run para verificar pipeline
!python -m src.trainer --config configs/grpo_config.yaml --start-iter 0 2>&1 | head -30

## 7. Agente autonomo

Treina Ministral-8B em loop. Metrica: **correct_rate** (maior = melhor).

Cada experimento = ~5 iteracoes de GRPO (~30-60 min no T4).

In [ ]:
%%bash
export PATH="/usr/local/bin:/usr/bin:/bin:/root/.local/bin:$PATH"
export CLAUDE_CODE_OAUTH_TOKEN=$(python3 -c "from google.colab import userdata; print(userdata.get('CLAUDE_CODE_OAUTH_TOKEN'))")
export HF_TOKEN=$(python3 -c "from google.colab import userdata; print(userdata.get('HF_TOKEN'))")
cd /content/codeforge-asm

claude -p "\
Hi! Have a look at program.md and let's kick off a new experiment. \
Do the setup first (create branch, read files, verify data, run prepare.py, init results.tsv), \
confirm, then start the autonomous experiment loop." \
--allowedTools "Bash Edit Write Read Glob Grep"

## 8. Resultados

In [ ]:
import pandas as pd

try:
    df = pd.read_csv("/content/codeforge-asm/results.tsv", sep="\t")
    keeps = df[df["status"] == "keep"]
    print(f"{len(df)} experimentos | {len(keeps)} kept | melhor correct_rate: {keeps['correct_rate'].max():.6f}\n")
    print(df.to_string(index=False))
except FileNotFoundError:
    print("Rode o agente primeiro.")

In [ ]:
# Salvar resultados
from google.colab import files
files.download("/content/codeforge-asm/results.tsv")